### Setup

In [0]:
%sql
--Create Catalog
CREATE CATALOG IF NOT EXISTS cars_catalog;

--Create Schema
CREATE SCHEMA IF NOT EXISTS cars_catalog.silver;
CREATE SCHEMA IF NOT EXISTS cars_catalog.gold;
CREATE SCHEMA IF NOT EXISTS cars_catalog.bronze;


### Data Reading


In [0]:
df = spark.read.format('parquet')\
            .option('inferSchema',True)\
            .load('abfss://bronze@cacdyamadls.dfs.core.windows.net/raw_data')

In [0]:
df.display()

### Removing Duplicates

In [0]:
df = df.dropDuplicates()

In [0]:
df.display()

###Data Transformation

In [0]:
from pyspark.sql.functions import *

from pyspark.sql.types import *

In [0]:
df = df.withColumn('model_category',split(col('Model_ID'),'-')[0])
df.display()

In [0]:
df.withColumn('Units_Sold',col('Units_Sold').cast(StringType())).printSchema()

In [0]:
df = df.withColumn('RevPerUnit',col('Revenue')/col("Units_Sold"))
display(df.select("RevPerUnit"))

### Aggregation

In [0]:
df.groupBy('Year', 'BranchName') \
  .agg(sum('Units_Sold').alias('Total_Units')) \
  .sort('Year', 'Total_Units', ascending=[1,0]) \
  .display()

Databricks visualization. Run in Databricks to view.

### Writing to Silver layer

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@cacdyamadls.dfs.core.windows.net/carsales')\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@cacdyamadls.dfs.core.windows.net/carsales`